# 04 — Train the disease detector (4 classes)

Kaggle GPU notebook, identical split to the local build (seed 42).

**The imbalance problem:** Caries 2,189 boxes vs Periapical Lesion 158
(13.9:1). v1 left this unhandled and its Caries AP@0.5 was 28.2%, with
Periapical almost never detected. This notebook applies:

1. **Image-level oversampling** — images containing Periapical Lesion are
   repeated 4x, Deep Caries 2x, so rare classes are seen more often. This
   brings the effective imbalance from 13.9:1 down to ~7:1.
2. **Copy-free augmentation** — flips are fine here (disease labels carry no
   position), so `fliplr=0.5` and mosaic are enabled.

Expected: 598 train / 107 val images, 3,529 lesion boxes.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# No internet on Kaggle without phone verification, so install from the
# private wheels dataset and use the bundled pretrained weights.
# Mount layout varies: /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
WHEELS = next(Path("/kaggle/input").rglob("yolov8s.pt")).parent
os.environ["YOLO_OFFLINE"] = "True"

try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                    "--find-links", str(WHEELS), "ultralytics", "ultralytics-thop"],
                   check=True)
    import ultralytics

import torch
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available. Kaggle silently runs CPU-only until the account is "
        "phone-verified: https://www.kaggle.com/settings -> Phone Verification."
    )
p = torch.cuda.get_device_properties(0)
print("gpu:", p.name, f"{p.total_memory / 1e9:.1f} GB")

In [ ]:
import glob
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SEED, VAL_FRAC = 42, 0.15

DISEASE_NAMES = ["Impacted", "Caries", "Periapical Lesion", "Deep Caries"]


def load_coco(path):
    with open(path) as fh:
        return json.load(fh)


def image_map(coco):
    return {i["id"]: i for i in coco["images"]}


def yolo_labels(coco, class_fn):
    imgs = image_map(coco)
    out = defaultdict(list)
    for ann in coco["annotations"]:
        img = imgs[ann["image_id"]]
        W, H = img["width"], img["height"]
        x, y, w, h = ann["bbox"]
        x1, y1 = max(0.0, x), max(0.0, y)
        x2, y2 = min(W, x + w), min(H, y + h)
        w, h = x2 - x1, y2 - y1
        if w <= 1 or h <= 1:
            continue
        out[img["file_name"]].append(
            f"{class_fn(ann)} {(x1 + w / 2) / W:.6f} {(y1 + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
    return dict(out)


def stratified_split(names, signature_fn, val_frac=VAL_FRAC, seed=SEED):
    groups = defaultdict(list)
    for n in names:
        groups[signature_fn(n)].append(n)
    rng = random.Random(seed)
    train, val = [], []
    for key in sorted(groups, key=str):
        members = sorted(groups[key])
        rng.shuffle(members)
        n_val = max(1, round(len(members) * val_frac)) if len(members) > 1 else 0
        val.extend(members[:n_val])
        train.extend(members[n_val:])
    return sorted(train), sorted(val)


def build(out_root, name, image_src, labels, train, val, class_names):
    root = Path(out_root) / name
    for split in ("train", "val"):
        (root / "images" / split).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split).mkdir(parents=True, exist_ok=True)
    for split, names in (("train", train), ("val", val)):
        for fname in names:
            src = Path(image_src) / fname
            dst = root / "images" / split / fname
            if not dst.exists():
                dst.symlink_to(src)
            rows = labels.get(fname, [])
            (root / "labels" / split / (Path(fname).stem + ".txt")).write_text(
                "\n".join(rows) + ("\n" if rows else ""))
    yaml = f"path: {root.resolve()}\ntrain: images/train\nval: images/val\nnames:\n"
    yaml += "".join(f"  {i}: {n}\n" for i, n in enumerate(class_names))
    (root / "data.yaml").write_text(yaml)
    return root

In [ ]:
# ---- locate the dataset and build the disease dataset ----
qed_json = next(INPUT.rglob("train_quadrant_enumeration_disease.json"))
qed = load_coco(qed_json)
qed_imgs = qed_json.parent / "xrays"
print("annotations:", qed_json)
print("images:", len(qed["images"]), "| boxes:", len(qed["annotations"]))

labels = yolo_labels(qed, lambda a: a["category_id_3"])

by_image = defaultdict(set)
for ann in qed["annotations"]:
    by_image[ann["image_id"]].add(ann["category_id_3"])
id_to_name = {i["id"]: i["file_name"] for i in qed["images"]}

def signature(name):
    ids = next((v for k, v in by_image.items() if id_to_name[k] == name), set())
    return tuple(sorted(ids))

train_names, val_names = stratified_split([i["file_name"] for i in qed["images"]], signature)
root = build(WORK / "datasets", "disease", qed_imgs, labels, train_names, val_names, DISEASE_NAMES)

n_train_boxes = sum(len(labels.get(n, [])) for n in train_names)
n_val_boxes = sum(len(labels.get(n, [])) for n in val_names)
print(f"train {len(train_names)} imgs / {n_train_boxes} boxes")
print(f"val   {len(val_names)} imgs / {n_val_boxes} boxes")
assert len(train_names) == 598 and len(val_names) == 107, "split drifted from the local build"
assert n_train_boxes + n_val_boxes == 3529, "box count drifted"
print("split matches the local build exactly.")

In [ ]:
# ---- image-level oversampling for the rare classes ----
name_to_classes = {id_to_name[k]: v for k, v in by_image.items()}

weights = {}
for name in train_names:
    classes = name_to_classes.get(name, set())
    w = 1
    if 2 in classes:      # Periapical Lesion (158 boxes)
        w = max(w, 4)
    if 3 in classes:      # Deep Caries (578 boxes)
        w = max(w, 2)
    weights[name] = w

repeated = [n for n in train_names for _ in range(weights[n])]
print(f"oversampled train list: {len(repeated)} entries from {len(train_names)} images "
      f"({len(repeated) / len(train_names):.2f}x)")

# effective class balance after oversampling
before = Counter()
after = Counter()
for ann in qed["annotations"]:
    fname = id_to_name[ann["image_id"]]
    if fname in weights:
        before[ann["category_id_3"]] += 1
        after[ann["category_id_3"]] += weights[fname]
print(f"{'class':22s} {'before':>7s} {'after':>7s}")
for cid, cname in enumerate(DISEASE_NAMES):
    print(f"{cname:22s} {before[cid]:7d} {after[cid]:7d}")
print(f"imbalance after oversampling: {max(after.values()) / max(1, min(after.values())):.1f}x")

list_path = root / "train_oversampled.txt"
list_path.write_text("\n".join(str((root / "images/train" / n).resolve()) for n in repeated) + "\n")
# point the dataset at the oversampled list
yaml = (root / "data.yaml").read_text().replace("train: images/train", f"train: {list_path}")
(root / "data.yaml").write_text(yaml)
print("\n" + yaml)

In [ ]:
from ultralytics import YOLO

IMGSZ = 1024
EPOCHS = 100
BATCH = 8
MODEL = str(WHEELS / "yolov8s.pt")

model = YOLO(MODEL)
results = model.train(
    data=str(root / "data.yaml"),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=25,
    seed=SEED,
    deterministic=True,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    workers=2,
    project=str(WORK / "runs"),
    name="disease_v2",
    exist_ok=True,
    fliplr=0.5,          # safe: disease labels are position-independent
    flipud=0.0,
    mosaic=0.5,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.3,
)
print("best weights:", results.save_dir)

In [ ]:
# ---- validation metrics ----
m = model.val(data=str(root / "data.yaml"), imgsz=IMGSZ, split="val", conf=0.001, iou=0.6)
names = [m.names[i] for i in sorted(m.names)]
ap50 = m.box.ap50.tolist()

print(f"mAP50    {m.box.map50:.4f}")
print(f"mAP50-95 {m.box.map:.4f}")
print(f"precision {m.box.mp:.4f} | recall {m.box.mr:.4f}")
print()
for name, ap in sorted(zip(names, ap50), key=lambda t: -t[1]):
    print(f"  {name:22s} AP50 {ap:.3f}")

summary = {
    "model": MODEL, "imgsz": IMGSZ, "epochs": EPOCHS, "batch": BATCH,
    "train_images": len(train_names), "val_images": len(val_names),
    "oversampling": {k: v for k, v in weights.items() if v > 1},
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
    "precision": float(m.box.mp), "recall": float(m.box.mr),
    "ap50_per_class": dict(zip(names, map(float, ap50))),
}
(WORK / "disease_metrics.json").write_text(json.dumps(summary, indent=2))
print("\nwrote disease_metrics.json")

In [ ]:
# ---- package outputs ----
import shutil
best = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best, WORK / "disease_best.pt")
shutil.make_archive(str(WORK / "disease_run"), "zip", str(results.save_dir))
print("disease_best.pt:", (WORK / "disease_best.pt").stat().st_size / 1e6, "MB")
print("artifacts in /kaggle/working: disease_best.pt, disease_metrics.json, disease_run.zip")

## Reading the results

Compare against the v1 baseline on the locked 50-image set:

| Metric | v1 |
|---|---|
| mAP@0.5 | 49.8% |
| Caries AP@0.5 | 28.2% |
| Periapical AP@0.5 | 42.9% (only 2 correct of 9) |

If Periapical is still weak after oversampling, raise its repeat factor to 6-8x
or switch to a two-stage crop-then-classify design in a follow-up run.